# Árboles de Decisión — Análisis del Dataset de Cáncer de Seno

Este notebook aplica los conceptos del capítulo de árboles de decisión a un problema real de diagnóstico médico. El objetivo es ir más allá de la mecánica del modelo y desarrollar una **lectura crítica** de las importancias de variables en un contexto donde las variables tienen significado clínico concreto.

## El Dataset: Wisconsin Breast Cancer

El [dataset de cáncer de seno de Wisconsin](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) contiene medidas computarizadas de biopsias de tumores de seno.

### ¿Qué se está midiendo exactamente?

El proceso de recolección de datos es el siguiente:

1. A una paciente se le realiza una **biopsia por aspiración con aguja fina (FNA)** del tumor — se extrae una pequeña muestra de tejido.
2. Esa muestra se observa bajo microscopio y se digitaliza la imagen.
3. Un algoritmo identifica los **núcleos celulares** visibles en la imagen y mide sus propiedades geométricas.

Es importante distinguir: **no se mide el tumor en sí**, sino los núcleos de las células que lo componen. Esto es relevante porque uno de los criterios histopatológicos clásicos del cáncer es la **atipia nuclear**: las células malignas tienen núcleos anormales — más grandes, con contornos irregulares, asimétricos y con más invaginaciones — comparados con células normales o benignas. El modelo aprende a detectar esa irregularidad.

### Variables del dataset

Para cada muestra se miden 10 características de los núcleos celulares:

| Característica | Qué describe |
|---|---|
| Radio, perímetro, área | Tamaño del núcleo |
| Textura | Variación en los niveles de gris de la imagen (irregularidad superficial) |
| Suavidad | Variación local en la longitud de los radios |
| Compacidad, concavidad, puntos cóncavos | Irregularidad del contorno |
| Simetría | Qué tan simétrico es el núcleo respecto a su eje mayor |
| Dimensión fractal | Complejidad del contorno (*coastline approximation*) |

Cada característica se reporta en tres versiones: **media**, **error estándar** y **peor valor** (promedio de los tres valores más extremos entre todas las células de la muestra), dando **30 variables en total**.

La variable objetivo es binaria: `1 = benigno`, `0 = maligno`.

**Pregunta central:** ¿Qué características de los núcleos celulares distinguen mejor un tumor benigno de uno maligno?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# Cargar datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnostico'] = df['target'].map({1: 'Benigno', 0: 'Maligno'})

print(f"Dimensiones: {df.shape}")
print(f"Clases: {df['diagnostico'].value_counts().to_dict()}")

## 1. Análisis Exploratorio

Antes de entrenar cualquier modelo, vale la pena entender qué tan separables son las dos clases en las variables más intuitivas.

In [ ]:
# Distribuciones de las variables 'mean' para las dos clases
mean_features = [c for c in data.feature_names if 'mean' in c]

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.flatten()

for i, feat in enumerate(mean_features):
    for label, color in [('Benigno', 'steelblue'), ('Maligno', 'tomato')]:
        vals = df[df['diagnostico'] == label][feat]
        axes[i].hist(vals, bins=25, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(feat.replace(' (mean)', '').replace('mean ', ''), fontsize=9)
    axes[i].set_yticks([])
    if i == 0:
        axes[i].legend(fontsize=8)

fig.suptitle('Distribución de características (media) por diagnóstico', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot de las dos variables más informativas visualmente
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (x_feat, y_feat) in zip(axes, [
    ('mean radius', 'mean concave points'),
    ('mean texture', 'mean area')
]):
    for label, color, marker in [('Benigno', 'steelblue', 'o'), ('Maligno', 'tomato', 'x')]:
        sub = df[df['diagnostico'] == label]
        ax.scatter(sub[x_feat], sub[y_feat], c=color, marker=marker,
                   alpha=0.5, s=25, label=label)
    ax.set_xlabel(x_feat, fontsize=10)
    ax.set_ylabel(y_feat, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Separabilidad entre clases en dos pares de variables', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Multicolinealidad entre Variables

El dataset tiene 30 variables pero muchas miden aspectos relacionados del mismo núcleo celular. Antes de interpretar importancias, conviene visualizar la estructura de correlación: variables altamente correlacionadas **comparten** importancia y ninguna de las dos aparece tan relevante como realmente es.

In [ ]:
# Matriz de correlación solo para variables 'mean'
corr = df[mean_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
# Limpiar nombres para que quepan
labels = [c.replace('mean ', '') for c in mean_features]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, rotation=0, fontsize=9)
ax.set_title('Correlación entre características (media)', fontsize=12)
plt.tight_layout()
plt.show()

# Pares con correlación > 0.9
print("Pares con |correlación| > 0.90:")
print("-" * 50)
for i in range(len(mean_features)):
    for j in range(i+1, len(mean_features)):
        c = corr.iloc[i, j]
        if abs(c) > 0.90:
            print(f"  {mean_features[i]:<25}  {mean_features[j]:<25}  r = {c:.3f}")

**Observación:** Radio, perímetro y área están correlacionadas casi perfectamente ($r > 0.98$) — son transformaciones geométricas del mismo concepto. Cuando el árbol elija una para dividir, la importancia de las otras dos caerá artificialmente. Esto lo veremos reflejado en las importancias.

## 3. Entrenamiento del Árbol

Entrenamos con las 30 variables y seleccionamos profundidad con validación cruzada.

In [ ]:
X = data.data
y = data.target
feat_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Selección de profundidad por CV
depths = range(2, 12)
cv_means, cv_stds = [], []
for d in depths:
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=d, random_state=42),
        X_train, y_train, cv=5, scoring='accuracy'
    )
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

best_depth = depths[np.argmax(cv_means)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, cv_means, marker='o', linewidth=2, color='steelblue')
ax.fill_between(depths,
                np.array(cv_means) - np.array(cv_stds),
                np.array(cv_means) + np.array(cv_stds),
                alpha=0.2, color='steelblue')
ax.axvline(x=best_depth, color='red', linestyle='--', linewidth=1.5,
           label=f'Mejor profundidad: {best_depth}')
ax.set_xlabel('Profundidad máxima', fontsize=11)
ax.set_ylabel('Accuracy (CV-5)', fontsize=11)
ax.set_title('Selección de profundidad por validación cruzada', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mejor profundidad: {best_depth}  |  CV accuracy: {max(cv_means):.3f}")

In [ ]:
# Modelo final
tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Maligno', 'Benigno'],
    colorbar=False, ax=ax
)
ax.set_title('Matriz de confusión — Test set', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Importancia MDI

La importancia por reducción de impureza se calcula internamente durante el entrenamiento. Recordar que suma exactamente 1 y está **ponderada por el tamaño del nodo**, lo que favorece divisiones en los niveles altos del árbol.

In [ ]:
mdi_imp = tree.feature_importances_
mdi_ord = np.argsort(mdi_imp)[::-1]

# Mostrar solo las 15 más importantes para legibilidad
top_n = 15
top_idx = mdi_ord[:top_n]

# Colorear por grupo de características
def feat_color(name):
    if 'mean' in name:   return 'steelblue'
    if 'error' in name:  return 'mediumseagreen'
    return 'tomato'  # worst

colors = [feat_color(feat_names[i]) for i in top_idx]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(top_n), mdi_imp[top_idx], color=colors, alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels([feat_names[i] for i in top_idx], fontsize=9)
ax.set_xlabel('Importancia MDI', fontsize=11)
ax.set_title(f'Top {top_n} variables — Importancia MDI\n(azul=media, verde=error_std, rojo=peor)', fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
for i, imp in enumerate(mdi_imp[top_idx]):
    ax.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontsize=8)

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue',    alpha=0.85, label='Media (mean)'),
    Patch(facecolor='mediumseagreen', alpha=0.85, label='Error estándar (SE)'),
    Patch(facecolor='tomato',       alpha=0.85, label='Peor valor (worst)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nImportancia acumulada del top {top_n}: {mdi_imp[top_idx].sum():.3f}")
print(f"Variables con importancia = 0: {(mdi_imp == 0).sum()} de {len(feat_names)}")

**¿Qué vemos?** Notar que radio, perímetro y área probablemente aparecen con importancias distintas pese a estar altamente correlacionadas. El árbol eligió una para las divisiones principales y las otras quedaron relegadas, no porque sean menos relevantes clínicamente, sino porque aportan información redundante.

## 5. Permutation Importance

Evaluamos cuánto cae el desempeño al barajar cada variable en el conjunto de test. Las barras de error reflejan la variabilidad entre las 30 repeticiones.

In [ ]:
perm_res  = permutation_importance(
    tree, X_test, y_test,
    n_repeats=30, random_state=42, n_jobs=-1, scoring='accuracy'
)
perm_mean = perm_res.importances_mean
perm_std  = perm_res.importances_std
perm_ord  = np.argsort(perm_mean)[::-1]

top_perm = perm_ord[:top_n]
colors_p = [feat_color(feat_names[i]) for i in top_perm]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), perm_mean[top_perm],
        xerr=perm_std[top_perm],
        color=colors_p, alpha=0.85, capsize=4)
ax.set_yticks(range(top_n))
ax.set_yticklabels([feat_names[i] for i in top_perm], fontsize=9)
ax.set_xlabel('Caída en accuracy al permutar la variable', fontsize=11)
ax.set_title(f'Top {top_n} variables — Permutation Importance\n(azul=media, verde=error_std, rojo=peor)', fontsize=11)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Comparación MDI vs. Permutation Importance

Las dos métricas responden preguntas distintas:

- **MDI**: ¿cuánto usó el árbol esta variable durante el entrenamiento?
- **Permutation**: ¿cuánto empeora el modelo en datos nuevos si la variable desaparece?

Cuando el ranking de una variable cambia mucho entre las dos métricas, vale la pena preguntarse por qué. Las causas más comunes son:

- **Multicolinealidad**: la variable comparte información con otra — el árbol eligió la otra, pero ambas serían igual de útiles.
- **Sobreajuste**: el árbol aprendió a usar la variable en entrenamiento, pero ese patrón no generaliza a test.

In [ ]:
import pandas as pd

# Rankings
mdi_ranks  = {feat_names[mdi_ord[i]]:  i+1 for i in range(len(feat_names))}
perm_ranks = {feat_names[perm_ord[i]]: i+1 for i in range(len(feat_names))}

# Scatter MDI vs. Permutation Importance
fig, ax = plt.subplots(figsize=(8, 6))

colors_scatter = [feat_color(n) for n in feat_names]
ax.scatter(mdi_imp, perm_mean, c=colors_scatter, alpha=0.8, s=60, zorder=3)

# Etiquetar las 6 variables con mayor MDI
for i in mdi_ord[:6]:
    ax.annotate(
        feat_names[i].replace('mean ', '').replace(' (mean)', ''),
        (mdi_imp[i], perm_mean[i]),
        textcoords='offset points', xytext=(6, 3), fontsize=8
    )

ax.set_xlabel('Importancia MDI\n(calculada en entrenamiento)', fontsize=11)
ax.set_ylabel('Permutation Importance\n(caída en accuracy en test)', fontsize=11)
ax.set_title('MDI vs. Permutation Importance\ncada punto = una variable', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(handles=legend_elements, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

# Tabla de discrepancias como resumen complementario
tabla = pd.DataFrame({
    'variable':  feat_names,
    'mdi_rank':  [mdi_ranks[f]  for f in feat_names],
    'perm_rank': [perm_ranks[f] for f in feat_names],
}).assign(delta=lambda d: (d['perm_rank'] - d['mdi_rank']).abs())\
  .sort_values('delta', ascending=False)

## 7. El Efecto de la Multicolinealidad en la Importancia

Radio, perímetro y área son prácticamente la misma cantidad geométrica. Veamos cómo se distribuye la importancia entre ellas comparado con lo que pasaría si elimináramos las redundantes.

In [ ]:
# Grupo de variables redundantes: radio, perímetro, área (en sus 3 versiones)
redundantes = [c for c in feat_names if any(k in c for k in ['radius', 'perimeter', 'area'])]

print("Importancias del grupo radio/perímetro/área:")
print("-" * 55)
total_grupo = 0
for feat in redundantes:
    idx = list(feat_names).index(feat)
    imp = mdi_imp[idx]
    total_grupo += imp
    print(f"  {feat:<35}  MDI = {imp:.4f}")
print(f"  {'TOTAL GRUPO':<35}  MDI = {total_grupo:.4f}")

print()

# Experimento: quitar radius — si perimeter/area compensan con el mismo accuracy,
# confirma que las tres variables son intercambiables
keep_mask = np.array(['radius' not in n for n in feat_names])
X_red = X[:, keep_mask]
feat_red = feat_names[keep_mask]

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_red, y, test_size=0.25, random_state=42, stratify=y
)
tree_red = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
tree_red.fit(X_tr_r, y_tr_r)

acc_full = (tree.predict(X_test) == y_test).mean()
acc_red  = (tree_red.predict(X_te_r) == y_te_r).mean()
print(f"Accuracy con todas las variables (incluyendo radius):  {acc_full:.3f}")
print(f"Accuracy sin radius (perimeter y area compensan):      {acc_red:.3f}")

In [ ]:
# Comparación de importancias: modelo completo vs. sin radius
mdi_red = tree_red.feature_importances_
mdi_red_ord = np.argsort(mdi_red)[::-1]
top_n_comp = 15

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: modelo completo — top 15
top_full = mdi_ord[:top_n_comp]
axes[0].barh(range(top_n_comp), mdi_imp[top_full],
             color=[feat_color(feat_names[i]) for i in top_full], alpha=0.85)
axes[0].set_yticks(range(top_n_comp))
axes[0].set_yticklabels([feat_names[i] for i in top_full], fontsize=9)
axes[0].set_xlabel('Importancia MDI', fontsize=10)
axes[0].set_title('Modelo completo\n(30 variables, radius incluido)', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='x')
for i, imp in enumerate(mdi_imp[top_full]):
    if imp > 0.005:
        axes[0].text(imp + 0.003, i, f'{imp:.3f}', va='center', fontsize=8)

# Panel 2: modelo sin radius — top 15
top_red = mdi_red_ord[:top_n_comp]
axes[1].barh(range(top_n_comp), mdi_red[top_red],
             color=[feat_color(feat_red[i]) for i in top_red], alpha=0.85)
axes[1].set_yticks(range(top_n_comp))
axes[1].set_yticklabels([feat_red[i] for i in top_red], fontsize=9)
axes[1].set_xlabel('Importancia MDI', fontsize=10)
axes[1].set_title('Sin radius\n(27 variables, perimeter/area compensan)', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='x')
for i, imp in enumerate(mdi_red[top_red]):
    if imp > 0.005:
        axes[1].text(imp + 0.003, i, f'{imp:.3f}', va='center', fontsize=8)

fig.suptitle('Al quitar radius, ¿qué variable toma su lugar?', fontsize=12)
plt.tight_layout()
plt.show()

# Cambio de importancia en perimeter y area al quitar radius
print("Cambio de importancia en perimeter/area al quitar radius:")
print("-" * 65)
for feat in [f for f in feat_red if 'perimeter' in f or 'area' in f]:
    idx_full = list(feat_names).index(feat)
    idx_red  = list(feat_red).index(feat)
    delta = mdi_red[idx_red] - mdi_imp[idx_full]
    signo = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
    print(f"  {feat:<35}  {mdi_imp[idx_full]:.4f}  →  {mdi_red[idx_red]:.4f}  ({signo})")

## 8. Visualización del Árbol y Lectura de las Reglas

Con profundidad limitada (3 niveles) podemos leer el árbol completo y verificar que las divisiones tienen sentido clínico.

In [ ]:
# Árbol legible de profundidad 3
tree_legible = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_legible.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    tree_legible,
    feature_names=feat_names,
    class_names=['Maligno', 'Benigno'],
    filled=True, rounded=True, fontsize=8,
    impurity=False, proportion=True, ax=ax
)
ax.set_title(
    'Árbol de decisión (profundidad 3) — Diagnóstico de cáncer de seno\n'
    'Color azul = mayoritariamente Benigno | Naranja = mayoritariamente Maligno',
    fontsize=11
)
plt.tight_layout()
plt.show()

acc_3 = (tree_legible.predict(X_test) == y_test).mean()
print(f"Accuracy del árbol de profundidad 3: {acc_3:.3f}")
print(f"(vs. profundidad {best_depth} óptima:    {acc_full:.3f})")

## 9. Resumen e Interpretación Clínica

### ¿Qué aprendimos del modelo?

1. **Variables más discriminativas**: Las características relacionadas con la *forma del contorno* (`concave points`, `concavity`) y el *tamaño* (`radius`, `area`) del núcleo celular son las que más diferencian tumores benignos de malignos. Esto es consistente con la literatura médica: los tumores malignos tienden a tener núcleos más grandes, irregulares y con más invaginaciones en el contorno.

2. **El efecto de la multicolinealidad**: Radio, perímetro y área miden esencialmente lo mismo desde perspectivas geométricas distintas. El árbol elige una y las otras quedan con importancia residual, pero eliminarlas no degrada el accuracy — son información redundante, no información adicional.

3. **MDI vs. Permutation**: Donde ambas métricas coinciden, la conclusión es robusta. Donde difieren, la causa más probable es la redundancia entre variables (como la tríada radio/perímetro/área).

4. **Trade-off interpretabilidad/accuracy**: El árbol de profundidad 3 es completamente legible y alcanza un accuracy comparable al árbol óptimo. En un contexto médico esto es valioso: el clínico puede auditar cada decisión del modelo.

### Limitaciones del árbol de decisión en este contexto

- **Inestabilidad**: pequeños cambios en los datos de entrenamiento pueden generar árboles muy distintos. Para producción médica, un Random Forest o Gradient Boosting daría estimaciones de importancia más estables.
- **Falsos negativos**: en diagnóstico oncológico, un maligno clasificado como benigno es mucho más costoso que el error inverso. El árbol no optimiza este trade-off por defecto — habría que ajustar el umbral de decisión o los pesos de clase.